# Dataset exploration
Run **after** `python -m training.prepare_data`. Open this notebook from the project root in PyCharm (or set the working directory to the project root).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
from PIL import Image
from src.data import load_splits, split_summary

## 1. Split table and class balance

In [ ]:
df = load_splits()
summary = split_summary(df)
print('Total images:', summary['total_images'])
print('Per split :', summary['per_split'])
df.groupby(['label', 'split']).size().unstack(fill_value=0)

## 2. Sample images per class (training split)

In [ ]:
classes = summary['classes']
fig, axes = plt.subplots(len(classes), 5, figsize=(12, 2.4 * len(classes)))
for r, cls in enumerate(classes):
    sample = df[(df.label == cls) & (df.split == 'train')].sample(5, random_state=0)
    for c, path in enumerate(sample['abs_path']):
        axes[r][c].imshow(Image.open(path))
        axes[r][c].axis('off')
        if c == 0:
            axes[r][c].set_title(cls, loc='left')
plt.tight_layout()
plt.show()

## 3. Image sizes (are they consistent?)

In [ ]:
sizes = [Image.open(p).size for p in df['abs_path'].sample(min(300, len(df)), random_state=0)]
widths, heights = zip(*sizes)
print('width  min/median/max:', min(widths), sorted(widths)[len(widths)//2], max(widths))
print('height min/median/max:', min(heights), sorted(heights)[len(heights)//2], max(heights))
print('=> all images are resized to 224x224 before entering the CNN.')